## Dataset

In [2]:
## Download necessary data files
!mkdir -p research/data
!cd research/data
!python -m pip install gdown
!gdown --folder "https://drive.google.com/drive/folders/1j21cc7-97TedKh_El5E34yI8o5ckI7eK"

# # remove the ones we dont need
# !rm -rf data/pdbbind_v2016
# !rm -rf data/pdbbind_v2020
# !rm data/test_set.zip
# !rm data/crossdocked_v1.1_rmsd1.0.tar.gz

Retrieving folder contents
Retrieving folder 1sp0HwdveKNpQoiQfdDOCPd1gE5rJWrmn pdbbind_v2016
Processing file 1cAN9KlVhbzj7tOSkJCNlHsDs82MeAu9j coreset.zip
Retrieving folder 1h885QFpEDQn7PP-c5ml5D6ZG-PpxwxFg pdbbind_v2020
Processing file 1IQ_a1S92kTwaIZKJgPeK0l8GUr-yXCkh pdbbind_v2020_all_match_emb.pt
Processing file 1oJX2th-TJXbf7L_zp4WDvfRRBF_qBo_v timesplit_no_lig_overlap_val
Processing file 1lUk5ySw3pfIvltDWVyJt39XZrD7qBhmo timesplit_test
Processing file 1BoKFqffFBsdhfukI6sJ4AFEI9giAZy9n crossdocked_pocket10_pose_split.pt
Processing file 1CUEh7HRaiagqDZ2ZyQxes49dQp9YUXaP crossdocked_v1.1_rmsd1.0_pocket10_processed_final.lmdb
Processing file 1T9jyEv7wq0nzn_G4JHyTQeevG5ULX8a6 crossdocked_v1.1_rmsd1.0.tar.gz
Processing file 1jzvUEzGDkC0WrsqrUsX6cYkl1P8rCFJJ split_by_name.pt
Processing file 1kfDqOxXKVS69VDNbyflSVDlBuR4t5iyW test_set.zip
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.g

## Create Train/Val/Test Split
The provided dataset from DiffDock has only train and test split. We need to make a validation split from train.

In [1]:
import pickle
import random
from pathlib import Path

import lmdb
import torch

lmdb_path = "data/crossdocked_v1.1_rmsd1.0_pocket10_processed_final.lmdb"
name_split_path = "data/split_by_name.pt"
out_path = "data/crossdocked_pose_split_from_name_val1000.pt"


def pair_key(pf, lf):
    return (str(pf), str(lf))


def pair_key_name(pf, lf):
    return (Path(str(pf)).name, Path(str(lf)).name)


name_split = torch.load(name_split_path, map_location="cpu")
train_pairs = [pair_key(p, l) for p, l in name_split["train"]]
test_pairs = [pair_key(p, l) for p, l in name_split["test"]]

wanted_full = set(train_pairs) | set(test_pairs)
wanted_name = set(pair_key_name(p, l) for p, l in wanted_full)

env = lmdb.open(
    lmdb_path,
    subdir=False,
    readonly=True,
    lock=False,
    readahead=False,
    max_readers=256,
)

mapping = {}
mapping_name = {}

with env.begin() as txn:
    cur = txn.cursor()
    for k, v in cur:
        rec = pickle.loads(v)
        pf = rec.get("protein_filename", "")
        lf = rec.get("ligand_filename", "")
        if not pf or not lf:
            continue

        key_full = pair_key(pf, lf)
        if key_full in wanted_full and key_full not in mapping:
            mapping[key_full] = int(k)

        key_name = pair_key_name(pf, lf)
        if key_name in wanted_name and key_name not in mapping_name:
            mapping_name[key_name] = int(k)

env.close()


def resolve_ids(pairs):
    ids = []
    misses = 0
    for pf, lf in pairs:
        idx = mapping.get((pf, lf))
        if idx is None:
            idx = mapping_name.get(pair_key_name(pf, lf))
        if idx is None:
            misses += 1
            continue
        ids.append(idx)
    return ids, misses


train_ids, train_miss = resolve_ids(train_pairs)
test_ids, test_miss = resolve_ids(test_pairs)

test_set = set(test_ids)
train_ids = [x for x in train_ids if x not in test_set]

seed = 0
val_n = 1000
rng = random.Random(seed)
rng.shuffle(train_ids)

val_ids = train_ids[:val_n]
train_ids = train_ids[val_n:]

if len(val_ids) != val_n:
    raise RuntimeError(f"not enough train ids for val={val_n}: got {len(val_ids)}")

out = {"train": train_ids, "val": val_ids, "test": test_ids}
torch.save(out, out_path)

print(
    "saved:", out_path,
    "\nsizes:", {k: len(out[k]) for k in out},
    "\nmisses:", {"train": train_miss, "test": test_miss},
)

split = torch.load("data/crossdocked_pose_split_from_name_val1000.pt")
assert not (set(split["train"]) & set(split["val"]) &set(split["test"]))

print("no id overlap")

saved: data/crossdocked_pose_split_from_name_val1000.pt 
sizes: {'train': 98990, 'val': 1000, 'test': 100} 
misses: {'train': 10, 'test': 0}
no id overlap


In [1]:
import os
from datamodules import CrossDockedDataModule

lmdb_path = os.path.abspath("data/crossdocked_v1.1_rmsd1.0_pocket10_processed_final.lmdb")
split_path = os.path.abspath("data/crossdocked_pose_split_from_name_val1000.pt")

dm = CrossDockedDataModule(
    lmdb_path=lmdb_path,
    split_pt_path=split_path,
    batch_size=8,
    num_workers=2,
)

batch = next(iter(dm.train_dataloader()))
print("protein_pos:", batch["protein_pos"].shape)
print("ligand_pos :", batch["ligand_pos"].shape)
print("bond_index :", batch["ligand_bond_index"].shape)
print("bond_type  :", batch["ligand_bond_type"].shape)
print("protein_counts:", batch["protein_counts"][:5].tolist())
print("ligand_counts :", batch["ligand_counts"][:5].tolist())
print("keys[0]:", batch["keys"][0])

# print everything within a sample
print("\n--- sample 0 ---")
for k in batch:
    print(f"{k}: {batch[k][0]}")
    break

protein_pos: torch.Size([3533, 3])
ligand_pos : torch.Size([206, 3])
bond_index : torch.Size([2, 450])
bond_type  : torch.Size([450])
protein_counts: [406, 425, 502, 358, 498]
ligand_counts : [25, 26, 33, 12, 22]
keys[0]: b'116191'

--- sample 0 ---
keys: b'116191'


In [3]:
import torch

def median_bond_len(batch):
    pos = batch["ligand_pos"].float()
    e = batch["ligand_bond_index"].long()
    a = pos[e[0]]
    b = pos[e[1]]
    d = (a - b).pow(2).sum(-1).sqrt()
    return float(d.median().item())

def pocket_scale(batch):
    pos = batch["protein_pos"].float()
    return float(torch.pdist(pos, p=2).median().item())

batch = next(iter(dm.train_dataloader()))
print("median bond length:", median_bond_len(batch))
print("median protein pdist:", pocket_scale(batch))


median bond length: 1.4188246726989746
median protein pdist: 14.135181427001953


In [ ]:
# trivial baselines for ligand position prediction from random time step in diffusion model

import torch
from tqdm import tqdm
from model.diffusion import center_by_protein
from model.diffusion import LigandDiffusion
from model.egnn import EGNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

denoiser = EGNN(
    num_layers=6,
    hidden_dim=256,
    edge_feat_dim=4,
    num_r_gaussian=16,
    k=32,
    cutoff_mode="knn",
    update_x=True,
    act_fn="silu",
    norm=False,
).to(device)

ligand_diffusion = LigandDiffusion(denoiser=denoiser, num_types=7).to(device)

@torch.inference_mode()
def pos_baselines(diffusion, loader, device):
    z = 0.0
    xtb = 0.0
    model = 0.0
    n = 0

    for batch in tqdm(loader):
        batch = {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}
        pb = batch["protein_batch"].long()
        lb = batch["ligand_batch"].long()
        bsz = int(pb.max().item()) + 1

        pp = batch["protein_pos"].float()
        lp = batch["ligand_pos"].float()
        pp = pp + diffusion.protein_noise_std * torch.randn_like(pp)
        pp, lp = center_by_protein(pp, lp, pb, lb, bsz)

        t_graph = torch.randint(0, diffusion.t, (bsz,), device=device)
        t_atom = (t_graph + 1)[lb]
        xt, _ = diffusion.q_sample_x(lp, t_atom)

        z += float(lp.pow(2).sum(-1).mean().item()) * bsz
        xtb += float((lp - xt).pow(2).sum(-1).mean().item()) * bsz
        # model += float(diffusion.loss(batch)["loss_position"].item()) * bsz
        n += bsz


    return {"zero": z / n, "xt": xtb / n} #"model": model / n}

print(pos_baselines(ligand_diffusion, dm.val_dataloader(), device))


  0%|          | 0/125 [00:00<?, ?it/s]

100%|██████████| 125/125 [00:02<00:00, 43.15it/s]

{'zero': 26.545380012512208, 'xt': 1.423866429567337}


In [ ]:
# finally, we can delete the leftover files we dont need anymore
!rm data/crossdocked_pocket10_pose_split.pt
!rm data/split_by_name.pt

In [2]:
from pathlib import Path
import numpy as np
from rdkit import Chem

AA_NAME_SYM = {
    "ALA": "A", "CYS": "C", "ASP": "D", "GLU": "E", "PHE": "F", "GLY": "G", "HIS": "H",
    "ILE": "I", "LYS": "K", "LEU": "L", "MET": "M", "ASN": "N", "PRO": "P", "GLN": "Q",
    "ARG": "R", "SER": "S", "THR": "T", "VAL": "V", "TRP": "W", "TYR": "Y",
}
CANONICAL_RESIDUES = set(AA_NAME_SYM.keys())


class PDBProtein:
    def __init__(self, data, mode="auto", skip_noncanonical=True):
        data = str(data)
        if (data.lower().endswith(".pdb") and mode == "auto") or mode == "path":
            with open(data, "r") as f:
                self.block = f.read()
        else:
            self.block = data

        self.skip_noncanonical = skip_noncanonical
        self.ptable = Chem.GetPeriodicTable()
        self.title = None
        self.atoms = []
        self.element = []
        self.atomic_weight = []
        self.pos = []
        self.atom_name = []
        self.residues = []
        self.parse()

    def enum_formatted_atom_lines(self):
        for line in self.block.splitlines():
            record = line[0:6].strip()
            if record == "ATOM":
                res_name = line[17:20].strip()
                if self.skip_noncanonical and res_name not in CANONICAL_RESIDUES:
                    continue

                element_symb = line[76:78].strip().capitalize()
                if len(element_symb) == 0:
                    element_symb = line[13:14]

                yield {
                    "line": line,
                    "type": "ATOM",
                    "atom_id": int(line[6:11]),
                    "atom_name": line[12:16].strip(),
                    "res_name": res_name,
                    "chain": line[21:22].strip(),
                    "res_id": int(line[22:26]),
                    "res_insert_id": line[26:27].strip(),
                    "x": float(line[30:38]),
                    "y": float(line[38:46]),
                    "z": float(line[46:54]),
                    "occupancy": float(line[54:60]),
                    "segment": line[72:76].strip(),
                    "element_symb": element_symb,
                    "charge": line[78:80].strip(),
                }
            elif record == "HEADER":
                yield {"type": "HEADER", "value": line[10:].strip()}
            elif record == "ENDMDL":
                break

    def parse(self):
        residues_tmp = {}
        for atom in self.enum_formatted_atom_lines():
            if atom["type"] == "HEADER":
                self.title = atom["value"].lower()
                continue

            atomic_number = self.ptable.GetAtomicNumber(atom["element_symb"])
            if atomic_number <= 0:
                continue

            atom_idx = len(self.element)
            self.atoms.append(atom)
            self.element.append(atomic_number)
            self.atomic_weight.append(self.ptable.GetAtomicWeight(atomic_number))
            self.pos.append(np.array([atom["x"], atom["y"], atom["z"]], dtype=np.float32))
            self.atom_name.append(atom["atom_name"])

            residue_key = f'{atom["chain"]}_{atom["segment"]}_{atom["res_id"]}_{atom["res_insert_id"]}'
            if residue_key not in residues_tmp:
                residues_tmp[residue_key] = {
                    "name": atom["res_name"],
                    "atoms": [atom_idx],
                    "chain": atom["chain"],
                    "segment": atom["segment"],
                }
            else:
                residues_tmp[residue_key]["atoms"].append(atom_idx)

        self.residues = [r for r in residues_tmp.values()]
        for residue in self.residues:
            sum_pos = np.zeros(3, dtype=np.float32)
            sum_mass = 0.0
            for atom_idx in residue["atoms"]:
                sum_pos += self.pos[atom_idx] * self.atomic_weight[atom_idx]
                sum_mass += self.atomic_weight[atom_idx]
            residue["center_of_mass"] = sum_pos / sum_mass

    def query_residues_ligand(self, ligand, radius, criterion="center_of_mass"):
        selected = []
        selected_indices = set()
        for ligand_atom_pos in ligand["pos"]:
            for residue_idx, residue in enumerate(self.residues):
                distance = np.linalg.norm(residue[criterion] - ligand_atom_pos, ord=2)
                if distance < radius and residue_idx not in selected_indices:
                    selected.append(residue)
                    selected_indices.add(residue_idx)
        return selected

    def residues_to_pdb_block(self, residues, name="POCKET"):
        block = f"HEADER    {name}\n"
        block += f"COMPND    {name}\n"
        for residue in residues:
            for atom_idx in residue["atoms"]:
                block += self.atoms[atom_idx]["line"] + "\n"
        block += "END\n"
        return block


def parse_sdf_positions(sdf_path):
    suppl = Chem.SDMolSupplier(str(sdf_path), removeHs=False, sanitize=True)
    mol = next((m for m in suppl if m is not None), None)
    if mol is None:
        raise ValueError(f"Could not read ligand from {sdf_path}")
    conf = mol.GetConformer()
    pos = conf.GetPositions().astype(np.float32)
    return {"pos": pos, "mol": mol}


def extract_targetdiff_pocket(protein_pdb_path, ligand_sdf_path, output_pdb_path, radius=10.0):
    protein = PDBProtein(protein_pdb_path, mode="path", skip_noncanonical=True)
    ligand = parse_sdf_positions(ligand_sdf_path)
    selected_residues = protein.query_residues_ligand(ligand, radius=radius, criterion="center_of_mass")

    if len(selected_residues) == 0:
        raise ValueError("No residues selected for the pocket. Check the protein/ligand coordinates and radius.")

    pocket_block = protein.residues_to_pdb_block(selected_residues, name=f"POCKET{int(radius)}")
    output_pdb_path = Path(output_pdb_path)
    output_pdb_path.write_text(pocket_block)

    print(f"Saved pocket PDB to: {output_pdb_path}")
    print(f"Selected residues: {len(selected_residues)}")
    print(f"Selected atoms: {sum(len(r['atoms']) for r in selected_residues)}")


folder = Path("data/pocket_examples")
protein_pdb_path = "data/test_set/ABL2_HUMAN_274_551_0/4xli_B_rec.pdb"
ligand_sdf_path = "data/test_set/ABL2_HUMAN_274_551_0/4xli_B_rec_4xli_1n1_lig_tt_min_0.sdf"
output_pdb_path = folder / "4xli_B_rec_pocket.pdb"

extract_targetdiff_pocket(
    protein_pdb_path=protein_pdb_path,
    ligand_sdf_path=ligand_sdf_path,
    output_pdb_path=output_pdb_path,
    radius=10.0,
)

Saved pocket PDB to: data/pocket_examples/4xli_B_rec_pocket.pdb
Selected residues: 51
Selected atoms: 396


In [3]:
from pathlib import Path
import shutil

import torch
import yaml

from config.config import InferenceConfig
from model.common import AtomCountPrior
from datamodules import CrossDockedDataModule

repo_root = Path("/blue/yanjun.li/ryan.hulke/BindHard")
ckpt_path = repo_root / "research/checkpoints/wgwnxpac/graphAttn_flow_matching_best.pt"
cfg_path = repo_root / "research/config/inference/flow_matching.yaml"

with open(cfg_path, "r") as f:
    cfg = InferenceConfig(**yaml.safe_load(f))

dm = CrossDockedDataModule(
    lmdb_path=cfg.lmdb_path,
    split_pt_path=cfg.split_path,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    persistent_workers=cfg.persistent_workers,
    prefetch_factor=cfg.prefetch_factor,
)

if hasattr(dm, "setup"):
    try:
        dm.setup(stage="fit")
    except TypeError:
        dm.setup()

train_ds = getattr(dm, "ds_train", None)
if train_ds is None:
    raise RuntimeError("dm.ds_train is missing after setup")

prior = AtomCountPrior.fit(train_ds, n_bins=10)

def prior_to_state_dict(prior_obj):
    if hasattr(prior_obj, "state_dict"):
        return prior_obj.state_dict()
    edges = prior_obj.edges.detach().cpu() if torch.is_tensor(prior_obj.edges) else prior_obj.edges
    return {
        "edges": edges,
        "values": tuple(prior_obj.values),
        "probs": tuple(prior_obj.probs),
    }

ckpt = torch.load(ckpt_path, map_location="cpu")
ckpt["prior"] = prior_to_state_dict(prior)

backup_path = ckpt_path.with_name(ckpt_path.stem + ".bak.pt")
if not backup_path.exists():
    shutil.copy2(ckpt_path, backup_path)

torch.save(ckpt, ckpt_path)

print(f"saved prior into: {ckpt_path}")
print(f"backup at: {backup_path}")
print("prior keys:", ckpt["prior"].keys())

saved prior into: /blue/yanjun.li/ryan.hulke/BindHard/research/checkpoints/wgwnxpac/graphAttn_flow_matching_best.pt
backup at: /blue/yanjun.li/ryan.hulke/BindHard/research/checkpoints/wgwnxpac/graphAttn_flow_matching_best.bak.pt
prior keys: dict_keys(['edges', 'values', 'probs'])


In [10]:
from config.config import InferenceConfig
from model.common import AtomCountPrior
from model.egnn import EGNN
from model.flow_matching import LigandFlowMatching
from __future__ import annotations

import json
import math
import os
import pickle
import tempfile
from pathlib import Path
from typing import Any

import modal
import torch
import yaml
from fastapi import Header, HTTPException
from openbabel import openbabel
from pydantic import BaseModel, Field
from rdkit import Chem
from rdkit.Chem import AllChem, QED, rdFingerprintGenerator, rdMolDescriptors
from vina import Vina

CONFIG_PATH = "/blue/yanjun.li/ryan.hulke/BindHard/research/config/inference/flow_matching.yaml"
CHECKPOINT_PATH = "/blue/yanjun.li/ryan.hulke/BindHard/research/checkpoints/wgwnxpac/graphAttn_flow_matching_best.pt"
AA_TO_INDEX_PATH = "/blue/yanjun.li/ryan.hulke/BindHard/app/backend/aa_to_index.json"
FPSCORES_PATH = "/blue/yanjun.li/ryan.hulke/BindHard/research/data/fpscores.pkl"

CANONICAL_RESIDUES = {
    "ALA", "ARG", "ASN", "ASP", "CYS",
    "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO",
    "SER", "THR", "TRP", "TYR", "VAL",
}

BACKBONE_ATOMS = ["CA", "C", "N", "O"]

ELEMENT_TO_ATOMIC_NUM = {
    "H": 1,
    "B": 5,
    "C": 6,
    "N": 7,
    "O": 8,
    "F": 9,
    "NA": 11,
    "MG": 12,
    "P": 15,
    "S": 16,
    "CL": 17,
    "K": 19,
    "CA": 20,
    "MN": 25,
    "FE": 26,
    "CO": 27,
    "NI": 28,
    "CU": 29,
    "ZN": 30,
    "SE": 34,
    "BR": 35,
    "I": 53,
}

DEFAULT_LIGAND_ELEMENTS = (6, 7, 8, 9, 15, 16, 17, 35, 53)
PERIODIC_TABLE = Chem.GetPeriodicTable()

# load aa_to_index.json into dict
aa_to_index = {}
with open(AA_TO_INDEX_PATH, "r") as f:
    raw = json.load(f)
    if not isinstance(raw, dict):
        raise RuntimeError("aa_to_index.json must contain a JSON object")
    for key, value in raw.items():
        aa = str(key).strip().upper()
        if aa not in CANONICAL_RESIDUES:
            raise RuntimeError(f"invalid residue in aa_to_index.json: {aa}")
        aa_to_index[aa] = int(value)


class GenerationRequest(BaseModel):
    pdb_text: str
    samples_per_target: int = Field(default=8, ge=1, le=64)
    return_trajectory: bool = False
    trajectory_stride: int = Field(default=1, ge=1, le=100)


class TrajectoryFrame(BaseModel):
    t: float
    ligand_pos: list[list[float]]
    ligand_type: list[int]
    ligand_atomic_nums: list[int]
    bonds: list[list[int]]


class SampleResponse(BaseModel):
    sample_idx: int
    status: str
    error: str | None
    n_atoms: int | None
    ligand_pos: list[list[float]] | None
    ligand_type: list[int] | None
    ligand_atomic_nums: list[int] | None
    trajectory: list[TrajectoryFrame] | None
    smiles: str | None
    vina_score: float | None
    qed_score: float | None
    sa_score: float | None


class SummaryResponse(BaseModel):
    n_samples: int
    n_valid: int
    n_invalid: int
    vina_mean: float | None
    qed_mean: float | None
    sa_mean: float | None


class GenerationResponse(BaseModel):
    samples: list[SampleResponse]
    summary: SummaryResponse


def load_aa_to_index(path: str = AA_TO_INDEX_PATH) -> dict[str, int]:
    raw = json.loads(Path(path).read_text(encoding="utf-8"))
    if not isinstance(raw, dict):
        raise RuntimeError("aa_to_index.json must contain a JSON object")
    out: dict[str, int] = {}
    for key, value in raw.items():
        aa = str(key).strip().upper()
        if aa not in CANONICAL_RESIDUES:
            raise RuntimeError(f"invalid residue in aa_to_index.json: {aa}")
        out[aa] = int(value)
    missing = sorted(CANONICAL_RESIDUES.difference(out))
    if missing:
        raise RuntimeError(f"aa_to_index.json is missing residues: {missing}")
    return out


def load_sa_fpscores() -> dict[int, float]:
    data = pickle.load(Path(FPSCORES_PATH).open("rb"))
    fpscores: dict[int, float] = {}
    for entry in data:
        for bit_id in entry[1:]:
            fpscores[int(bit_id)] = float(entry[0])
    return fpscores

fpscores = load_sa_fpscores()

def infer_element_from_atom_name(atom_name: str) -> str:
    letters = "".join(ch for ch in atom_name if ch.isalpha()).upper()
    if not letters:
        raise ValueError(f"cannot infer element from atom name '{atom_name}'")
    if len(letters) >= 2 and letters[:2] in ELEMENT_TO_ATOMIC_NUM:
        return letters[:2]
    return letters[0]


def parse_pocket_pdb_text(
    pdb_text: str,
    aa_to_index: dict[str, int],
    keep_hydrogens: bool = False,
) -> tuple[dict[str, torch.Tensor], torch.Tensor]:
    protein_pos: list[list[float]] = []
    protein_element: list[int] = []
    protein_atom_to_aa_type: list[int] = []
    protein_is_backbone: list[bool] = []

    residue_atom_indices: dict[str, list[int]] = {}
    residue_mass_sum: dict[str, float] = {}
    residue_weighted_pos_sum: dict[str, torch.Tensor] = {}

    saw_model = False

    for line_no, raw_line in enumerate(pdb_text.splitlines(), start=1):
        line = raw_line.rstrip("\n")
        record = line[0:6].strip().upper() if len(line) >= 6 else ""

        if record == "MODEL":
            saw_model = True
            continue
        if record == "ENDMDL" and saw_model:
            break
        if record != "ATOM":
            continue
        if len(line) < 54:
            raise ValueError(f"line {line_no}: ATOM record too short")

        altloc = line[16:17].strip().upper()
        if altloc not in {"", "A"}:
            continue

        resname = line[17:20].strip().upper()
        if resname not in aa_to_index:
            continue

        atom_name = line[12:16].strip().upper()
        chain = line[21:22].strip()
        res_id = line[22:26].strip()
        res_insert_id = line[26:27].strip()
        segment = line[72:76].strip() if len(line) >= 76 else ""

        try:
            x = float(line[30:38].strip())
            y = float(line[38:46].strip())
            z = float(line[46:54].strip())
        except ValueError as exc:
            raise ValueError(f"line {line_no}: invalid XYZ coordinates") from exc

        element = line[76:78].strip().upper() if len(line) >= 78 else ""
        if not element:
            element = infer_element_from_atom_name(atom_name)
        if element not in ELEMENT_TO_ATOMIC_NUM:
            raise ValueError(f"line {line_no}: unsupported element '{element}'")

        atomic_num = ELEMENT_TO_ATOMIC_NUM[element]
        if atomic_num == 1 and not keep_hydrogens:
            continue

        atom_index = len(protein_pos)

        protein_pos.append([x, y, z])
        protein_element.append(atomic_num)
        protein_atom_to_aa_type.append(aa_to_index[resname])
        protein_is_backbone.append(atom_name in BACKBONE_ATOMS)

        residue_key = f"{chain}_{segment}_{res_id}_{res_insert_id}"
        if residue_key not in residue_atom_indices:
            residue_atom_indices[residue_key] = []
            residue_mass_sum[residue_key] = 0.0
            residue_weighted_pos_sum[residue_key] = torch.zeros(3, dtype=torch.float32)

        residue_atom_indices[residue_key].append(atom_index)

        atom_mass = float(PERIODIC_TABLE.GetAtomicWeight(int(atomic_num)))
        residue_mass_sum[residue_key] += atom_mass
        residue_weighted_pos_sum[residue_key] += atom_mass * torch.tensor([x, y, z], dtype=torch.float32)

    if not protein_pos:
        raise ValueError("no protein ATOM records parsed from pdb_text")

    residue_centers: list[torch.Tensor] = []
    for residue_key in residue_atom_indices:
        total_mass = residue_mass_sum[residue_key]
        if total_mass <= 0.0:
            continue
        residue_centers.append(residue_weighted_pos_sum[residue_key] / total_mass)

    if not residue_centers:
        raise ValueError("no residue centers could be computed from pdb_text")

    n_atoms = len(protein_pos)
    protein_dict = {
        "protein_pos": torch.tensor(protein_pos, dtype=torch.float32),
        "protein_batch": torch.zeros(n_atoms, dtype=torch.long),
        "protein_element": torch.tensor(protein_element, dtype=torch.long),
        "protein_atom_to_aa_type": torch.tensor(protein_atom_to_aa_type, dtype=torch.long),
        "protein_is_backbone": torch.tensor(protein_is_backbone, dtype=torch.bool),
    }
    residue_centers_tensor = torch.stack(residue_centers, dim=0)
    return protein_dict, residue_centers_tensor

def compute_box_from_pocket_residues(residue_centers: torch.Tensor) -> tuple[list[float], list[float]]:
    residue_centers = residue_centers.detach().cpu().float()
    if residue_centers.ndim != 2 or residue_centers.shape[1] != 3 or residue_centers.shape[0] == 0:
        raise ValueError("residue_centers must have shape [N, 3] with N > 0")

    center = residue_centers.mean(dim=0).tolist()
    residue_span = (residue_centers.max(dim=0).values - residue_centers.min(dim=0).values).tolist()

    box_size = [max(22.0, float(v) - 8.0) for v in residue_span]
    return center, box_size

def summarize_box_fit(ligand_pos: torch.Tensor, box_center: list[float], box_size: list[float]) -> dict[str, Any]:
    pos = ligand_pos.detach().cpu().float()
    lig_min = pos.min(dim=0).values
    lig_max = pos.max(dim=0).values

    center = torch.tensor(box_center, dtype=torch.float32)
    size = torch.tensor(box_size, dtype=torch.float32)
    box_min = center - size / 2.0
    box_max = center + size / 2.0

    below = (box_min - lig_min).clamp(min=0)
    above = (lig_max - box_max).clamp(min=0)

    return {
        "lig_min": lig_min.tolist(),
        "lig_max": lig_max.tolist(),
        "lig_center": pos.mean(dim=0).tolist(),
        "box_min": box_min.tolist(),
        "box_max": box_max.tolist(),
        "box_center": center.tolist(),
        "box_size": size.tolist(),
        "outside_below": below.tolist(),
        "outside_above": above.tolist(),
    }

def decode_atomic_num(type_id: int, atom_type_decoder: dict[int, Any]) -> int:
    decoder = atom_type_decoder or {
        i: atomic_num for i, atomic_num in enumerate(DEFAULT_LIGAND_ELEMENTS)
    }
    if int(type_id) not in decoder:
        valid = sorted(int(x) for x in decoder.keys())
        raise ValueError(f"unknown ligand_type index {type_id}; valid indices are {valid}")

    value = decoder[int(type_id)]
    if isinstance(value, dict):
        atomic_num = value.get("atomic_num", value.get("atomic_number", value.get("element")))
    else:
        atomic_num = value

    if isinstance(atomic_num, str):
        atomic_num = PERIODIC_TABLE.GetAtomicNumber(atomic_num)
    atomic_num = int(atomic_num)
    if atomic_num <= 0:
        raise ValueError(f"invalid atomic number decoded from type {type_id}: {atomic_num}")
    return atomic_num


def build_ob_mol_from_geometry(
    ligand_pos: Any,
    ligand_type: Any,
    atom_type_decoder: dict[int, Any],
) -> openbabel.OBMol | None:
    if hasattr(ligand_pos, "detach"):
        ligand_pos = ligand_pos.detach()
    if hasattr(ligand_pos, "cpu"):
        ligand_pos = ligand_pos.cpu()
    if hasattr(ligand_pos, "tolist"):
        ligand_pos = ligand_pos.tolist()

    if hasattr(ligand_type, "detach"):
        ligand_type = ligand_type.detach()
    if hasattr(ligand_type, "cpu"):
        ligand_type = ligand_type.cpu()
    if hasattr(ligand_type, "tolist"):
        ligand_type = ligand_type.tolist()

    coords = [list(row) for row in ligand_pos]
    type_ids = [int(x) for x in ligand_type]

    if not coords or not type_ids:
        return None
    if len(coords) != len(type_ids):
        raise ValueError("ligand_pos and ligand_type must have same length")
    if not all(isinstance(row, (list, tuple)) and len(row) == 3 for row in coords):
        raise ValueError("ligand_pos must have shape [N, 3]")

    ob_mol = openbabel.OBMol()
    ob_mol.BeginModify()
    for xyz, type_id in zip(coords, type_ids):
        atom = ob_mol.NewAtom()
        atom.SetAtomicNum(decode_atomic_num(type_id, atom_type_decoder))
        atom.SetVector(float(xyz[0]), float(xyz[1]), float(xyz[2]))
    ob_mol.EndModify()
    ob_mol.ConnectTheDots()
    ob_mol.PerceiveBondOrders()
    return ob_mol


def infer_bonds_from_geometry(
    ligand_pos: Any,
    ligand_type: Any,
    atom_type_decoder: dict[int, Any],
) -> list[list[int]]:
    ob_mol = build_ob_mol_from_geometry(
        ligand_pos=ligand_pos,
        ligand_type=ligand_type,
        atom_type_decoder=atom_type_decoder,
    )
    if ob_mol is None:
        return []

    pair_to_order: dict[tuple[int, int], int] = {}
    for ob_bond in openbabel.OBMolBondIter(ob_mol):
        begin = int(ob_bond.GetBeginAtomIdx()) - 1
        end = int(ob_bond.GetEndAtomIdx()) - 1
        if begin < 0 or end < 0 or begin == end:
            continue
        a, b = (begin, end) if begin < end else (end, begin)
        order = max(1, int(ob_bond.GetBondOrder()))
        if (a, b) not in pair_to_order or order > pair_to_order[(a, b)]:
            pair_to_order[(a, b)] = order

    return [[a, b, order] for (a, b), order in sorted(pair_to_order.items())]


def infer_mol_from_geometry(
    ligand_pos: Any,
    ligand_type: Any,
    atom_type_decoder: dict[int, Any],
) -> Chem.Mol | None:
    ob_mol = build_ob_mol_from_geometry(
        ligand_pos=ligand_pos,
        ligand_type=ligand_type,
        atom_type_decoder=atom_type_decoder,
    )
    if ob_mol is None:
        return None

    conv = openbabel.OBConversion()
    conv.SetOutFormat("mol")
    mol_block = conv.WriteString(ob_mol)
    rd_mol = Chem.MolFromMolBlock(mol_block, sanitize=False, removeHs=False)
    if rd_mol is None:
        return None

    try:
        Chem.SanitizeMol(rd_mol)
    except Exception:
        try:
            Chem.SanitizeMol(rd_mol, Chem.SANITIZE_ALL ^ Chem.SANITIZE_KEKULIZE)
        except Exception:
            return None
    return rd_mol


def is_usable_mol(mol: Chem.Mol | None) -> bool:
    if mol is None:
        return False
    try:
        test_mol = Chem.Mol(mol)
        Chem.SanitizeMol(test_mol)
        if len(Chem.GetMolFrags(test_mol)) != 1:
            return False
        return True
    except Exception:
        return False


def protein_to_pdbqt_string(protein: dict[str, torch.Tensor]) -> str:
    protein_pos = protein["protein_pos"].detach().cpu().float()
    protein_element = protein["protein_element"].detach().cpu().long()
    if "protein_batch" in protein:
        mask = protein["protein_batch"].detach().cpu().long() == 0
        protein_pos = protein_pos[mask]
        protein_element = protein_element[mask]

    ob_rec = openbabel.OBMol()
    ob_rec.BeginModify()
    for xyz, z in zip(protein_pos.tolist(), protein_element.tolist()):
        atom = ob_rec.NewAtom()
        atom.SetAtomicNum(int(z))
        atom.SetVector(float(xyz[0]), float(xyz[1]), float(xyz[2]))
    ob_rec.EndModify()

    conv = openbabel.OBConversion()
    conv.SetOutFormat("pdbqt")
    conv.AddOption("r", openbabel.OBConversion.OUTOPTIONS)
    return conv.WriteString(ob_rec)


def mol_to_pdbqt_string(mol: Chem.Mol) -> str:
    work_mol = Chem.Mol(mol)
    Chem.RemoveStereochemistry(work_mol)
    work_mol = Chem.AddHs(work_mol, addCoords=True)

    if work_mol.GetNumConformers() == 0:
        status = AllChem.EmbedMolecule(work_mol, randomSeed=0)
        if status != 0:
            raise ValueError("rdkit embedding failed")

    conv = openbabel.OBConversion()
    conv.SetInAndOutFormats("mol", "pdbqt")
    ob_mol = openbabel.OBMol()
    ok = conv.ReadString(ob_mol, Chem.MolToMolBlock(work_mol))
    if not ok:
        raise ValueError("openbabel failed to read ligand mol block")
    return conv.WriteString(ob_mol)


def compute_qed(mol: Chem.Mol) -> float:
    return float(QED.qed(mol))


def compute_sa(mol: Chem.Mol, fpscores: dict[int, float]) -> float:
    sa_mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2)
    sparse_fp = sa_mfpgen.GetSparseCountFingerprint(mol)
    nze = sparse_fp.GetNonzeroElements()

    score1 = 0.0
    nf = 0
    for bit_id, count in nze.items():
        nf += count
        score1 += fpscores.get(int(bit_id), -4.0) * int(count)

    if nf == 0:
        return float("nan")

    score1 /= nf

    n_atoms = mol.GetNumAtoms()
    n_chiral_centers = len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))
    ring_info = mol.GetRingInfo()
    n_spiro = rdMolDescriptors.CalcNumSpiroAtoms(mol)
    n_bridgeheads = rdMolDescriptors.CalcNumBridgeheadAtoms(mol)
    n_macrocycles = sum(1 for ring in ring_info.AtomRings() if len(ring) > 8)

    size_penalty = n_atoms ** 1.005 - n_atoms
    stereo_penalty = math.log10(n_chiral_centers + 1)
    spiro_penalty = math.log10(n_spiro + 1)
    bridge_penalty = math.log10(n_bridgeheads + 1)
    macrocycle_penalty = math.log10(2) if n_macrocycles > 0 else 0.0

    score2 = (
        -size_penalty
        - stereo_penalty
        - spiro_penalty
        - bridge_penalty
        - macrocycle_penalty
    )

    score3 = 0.0
    num_bits = len(nze)
    if n_atoms > num_bits and num_bits > 0:
        score3 = math.log(float(n_atoms) / num_bits) * 0.5

    sa_score = score1 + score2 + score3

    raw_min = -4.0
    raw_max = 2.5
    sa_score = 11.0 - (sa_score - raw_min + 1.0) / (raw_max - raw_min) * 9.0

    if sa_score > 8.0:
        sa_score = 8.0 + math.log(sa_score - 8.0)

    if sa_score > 10.0:
        return 10.0
    if sa_score < 1.0:
        return 1.0
    return float(sa_score)


def smiles_from_mol(mol: Chem.Mol) -> str:
    mol_no_h = Chem.RemoveHs(Chem.Mol(mol))
    return str(Chem.MolToSmiles(mol_no_h, canonical=True))


def mean_or_none(values: list[float]) -> float | None:
    if not values:
        return None
    return float(sum(values) / len(values))

with Path(CONFIG_PATH).open("r", encoding="utf-8") as f:
    cfg = InferenceConfig(**yaml.safe_load(f))

atom_type_decoder = {
    i: int(z)
    for i, z in enumerate(DEFAULT_LIGAND_ELEMENTS[:cfg.num_types])
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

denoiser = EGNN(
    num_layers=cfg.num_layers,
    hidden_dim=cfg.hidden_dim,
    edge_feat_dim=cfg.edge_feat_dim,
    num_r_gaussian=cfg.num_r_gaussian,
    message_passing_mode=cfg.message_passing_mode,
    k=cfg.k,
    cutoff_mode=cfg.cutoff_mode,
    update_x=True,
    norm=cfg.norm,
).to(device)

model_kwargs: dict[str, Any] = {
    "denoiser": denoiser,
    "hidden_dim": cfg.hidden_dim,
    "num_types": cfg.num_types,
    "steps": cfg.steps,
}
if hasattr(cfg, "type_loss_scale"):
    model_kwargs["type_loss_scale"] = cfg.type_loss_scale
if hasattr(cfg, "protein_noise_std"):
    model_kwargs["protein_noise_std"] = cfg.protein_noise_std

model = LigandFlowMatching(**model_kwargs).to(device)

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
model.load_state_dict(ckpt["diffusion"], strict=True)
model.eval()
prior = AtomCountPrior.from_state_dict(ckpt["prior"])

def get_samples(pdb_text, samples_per_target, return_trajectory, trajectory_stride):
    try:
        protein_raw_cpu, residue_centers = parse_pocket_pdb_text(
            pdb_text=pdb_text,
            aa_to_index=aa_to_index,
            keep_hydrogens=False,
        )
        protein_shift = protein_raw_cpu["protein_pos"].mean(dim=0)

        protein_model_cpu = {
            key: value.clone() if torch.is_tensor(value) else value
            for key, value in protein_raw_cpu.items()
        }
        protein_model_cpu["protein_pos"] = protein_model_cpu["protein_pos"] - protein_shift

        box_center, box_size = compute_box_from_pocket_residues(residue_centers)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc

    protein_gpu = {
        key: value.to(device, non_blocking=True)
        for key, value in protein_model_cpu.items()
    }

    amp = device.type == "cuda"
    amp_dtype = torch.bfloat16 if amp else torch.float32

    with torch.inference_mode(), torch.autocast(
        device_type=device.type,
        dtype=amp_dtype,
        enabled=amp,
    ):
        sample = model.sample(
            protein_gpu,
            prior,
            num_samples=samples_per_target,
            return_trajectory=return_trajectory,
            trajectory_stride=trajectory_stride,
        )

    ligand_batch = sample["ligand_batch"].detach().cpu().long()
    ligand_pos = sample["ligand_pos"].detach().cpu()
    ligand_type = sample["ligand_type"].detach().cpu().long()
    traj = sample.get("trajectory")

    receptor_pdbqt = protein_to_pdbqt_string(protein_model_cpu)
    out_samples: list[SampleResponse] = []
    valid_vina: list[float] = []
    valid_qed: list[float] = []
    valid_sa: list[float] = []

    with tempfile.TemporaryDirectory() as temp_dir_str:
        temp_dir = Path(temp_dir_str)
        receptor_path = temp_dir / "receptor.pdbqt"
        receptor_path.write_text(receptor_pdbqt, encoding="utf-8")

        vina_engine = Vina(sf_name="vina")
        vina_engine.set_receptor(str(receptor_path))
        vina_engine.compute_vina_maps(
            center=[float(x) for x in box_center],
            box_size=[float(x) for x in box_size],
        )

        for local_idx in range(samples_per_target):
            lig_mask = ligand_batch == local_idx
            sample_pos_tensor = ligand_pos[lig_mask]
            sample_type_tensor = ligand_type[lig_mask]
            sample_atomic_nums = [
                decode_atomic_num(int(type_id), atom_type_decoder)
                for type_id in sample_type_tensor.tolist()
            ]
            try:
                fit_stats = summarize_box_fit(sample_pos_tensor, box_center, box_size)
                print(f"sample {local_idx} box fit:", fit_stats)
                mol = infer_mol_from_geometry(
                    ligand_pos=sample_pos_tensor,
                    ligand_type=sample_type_tensor,
                    atom_type_decoder=atom_type_decoder,
                )
                if not is_usable_mol(mol):
                    raise ValueError("invalid_rdkit_mol")

                print(f"sample {local_idx}: inferred mol with {mol.GetNumAtoms()} atoms and formula {Chem.rdMolDescriptors.CalcMolFormula(mol)}")
                mol = Chem.Mol(mol)
                Chem.SanitizeMol(mol)
                print(f"sample {local_idx}: sanitized mol; now has {mol.GetNumAtoms()} atoms")

                smiles = smiles_from_mol(mol)
                print(f"sample {local_idx}: canonical SMILES: {smiles}")
                
                ligand_pdbqt_path = temp_dir / f"ligand_{local_idx:04d}.pdbqt"
                ligand_pdbqt_path.write_text(mol_to_pdbqt_string(mol), encoding="utf-8")
                vina_engine.set_ligand_from_file(str(ligand_pdbqt_path))
                vina_score = float(vina_engine.score()[0])

                qed_score = compute_qed(mol)
                sa_score = compute_sa(mol, fpscores)

                valid_vina.append(vina_score)
                valid_qed.append(qed_score)
                valid_sa.append(sa_score)

                trajectory_payload: list[TrajectoryFrame] | None = None
                if traj is not None:
                    trajectory_payload = []
                    for frame in traj:
                        frame_pos = frame["ligand_pos"][lig_mask].detach().cpu()
                        frame_type = frame["ligand_type"][lig_mask].detach().cpu().long()
                        frame_atomic_nums = [
                            decode_atomic_num(int(type_id), atom_type_decoder)
                            for type_id in frame_type.tolist()
                        ]
                        trajectory_payload.append(
                            TrajectoryFrame(
                                t=float(frame["t"]),
                                ligand_pos=frame_pos.tolist(),
                                ligand_type=frame_type.tolist(),
                                ligand_atomic_nums=frame_atomic_nums,
                                bonds=infer_bonds_from_geometry(
                                    ligand_pos=frame_pos,
                                    ligand_type=frame_type,
                                    atom_type_decoder=atom_type_decoder,
                                ),
                            )
                        )

                out_samples.append(
                    SampleResponse(
                        sample_idx=local_idx,
                        status="completed",
                        error=None,
                        n_atoms=int(sample_type_tensor.shape[0]),
                        ligand_pos=sample_pos_tensor.tolist(),
                        ligand_type=sample_type_tensor.tolist(),
                        ligand_atomic_nums=sample_atomic_nums,
                        trajectory=trajectory_payload,
                        smiles=smiles,
                        vina_score=vina_score,
                        qed_score=qed_score,
                        sa_score=sa_score,
                    )
                )
            except Exception as exc:
                out_samples.append(
                    SampleResponse(
                        sample_idx=local_idx,
                        status="failed",
                        error=str(exc),
                        n_atoms=None,
                        ligand_pos=None,
                        ligand_type=None,
                        ligand_atomic_nums=None,
                        trajectory=None,
                        smiles=None,
                        vina_score=None,
                        qed_score=None,
                        sa_score=None,
                    )
                )

    summary = SummaryResponse(
        n_samples=len(out_samples),
        n_valid=sum(1 for row in out_samples if row.status == "completed"),
        n_invalid=sum(1 for row in out_samples if row.status != "completed"),
        vina_mean=mean_or_none(valid_vina),
        qed_mean=mean_or_none(valid_qed),
        sa_mean=mean_or_none(valid_sa),
    )

    return GenerationResponse(samples=out_samples, summary=summary).model_dump()

In [11]:
path = Path("/blue/yanjun.li/ryan.hulke/BindHard/research/data/pocket_examples/protein_pocket10.pdb")
payload = {
    "pdb_text": path.read_text(),
    "samples_per_target": 10,
    "return_trajectory": True,
    "trajectory_stride": 1,
}

In [12]:
res = get_samples(**payload)
print(json.dumps(res, indent=2))

Computing Vina grid ... done.
sample 0 box fit: {'lig_min': [-2.2739052772521973, -1.2713513374328613, -3.822507858276367], 'lig_max': [0.9751346707344055, -0.03024187497794628, 1.677243709564209], 'lig_center': [-0.6182042360305786, -0.7399613261222839, -0.7182928323745728], 'box_min': [-24.178335189819336, -18.519378662109375, 12.628379821777344], 'box_max': [-2.178335189819336, 3.4806203842163086, 34.628379821777344], 'box_center': [-13.178335189819336, -7.519379615783691, 23.628379821777344], 'box_size': [22.0, 22.0, 22.0], 'outside_below': [0.0, 0.0, 16.45088768005371], 'outside_above': [3.1534698009490967, 0.0, 0.0]}
sample 0: inferred mol with 9 atoms and formula C7NO
sample 0: sanitized mol; now has 9 atoms
sample 0: canonical SMILES: [N][C][C][C]1[C][C][C]([O])[C]1
sample 1 box fit: {'lig_min': [-2.066035747528076, -1.8239213228225708, -3.0242018699645996], 'lig_max': [0.7186331748962402, 0.296227365732193, 1.3881566524505615], 'lig_center': [-0.7978800535202026, -0.4775623679

[17:02:42] Both bonds on one end of an atropisomer are on the same side - atoms are: 9 2
[17:02:42] Both bonds on one end of an atropisomer are on the same side - atoms are: 4 9
